<a href="https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/14_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer

## Transformers: a quick introduction

The Transformer, introduced in the seminal paper ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) by Vaswani et al. (2017), replaces recurrence with attention and enables fully parallel training. It underpins most state‑of‑the‑art models in NLP and beyond, including BERT, GPT, and T5.

### Why it matters
- Self‑attention lets every token attend to every other token in a single layer, capturing long‑range dependencies without the vanishing‑gradient issues of RNNs.
- No time‑step loops → efficient GPU/TPU utilization and faster training.
- Scales predictably to very large models (billions of parameters).

### Core ideas
- Self‑attention computes context‑aware representations via Query, Key, Value projections and attention weights: $\text{softmax}\!\left(\tfrac{QK^\top}{\sqrt{d_k}}\right)V$.
- Positional encodings inject order information absent in pure attention.
- Multi‑head attention attends to multiple representation subspaces in parallel.
- Position‑wise feed‑forward networks add nonlinearity and capacity.
- Residual connections and layer normalization stabilize deep stacks.

### Where it’s used
- Machine translation, summarization, question answering
- Text/code generation and retrieval‑augmented generation
- Classification and information extraction
- Speech and vision transformers (ViT, DETR) extend the paradigm beyond text

### Model families at a glance
- Encoder‑only (BERT): bidirectional representations for understanding tasks.
- Decoder‑only (GPT): autoregressive generation.
- Encoder–decoder (T5): sequence‑to‑sequence with cross‑attention.

### In this notebook
- Build intuition for the architecture and attention mechanism
- Trace shapes and data flow you’ll see in practice
- Note practical tips for training and scaling


## Transformer Architecture

![Transformer Architecture](https://miro.medium.com/max/700/1*BHzGVskWGS_3jEcYYi6miQ.png)

The Transformer architecture consists of an encoder (left) and a decoder (right):

- **Encoder**: Processes the input sequence through multiple identical layers of self-attention and feed-forward networks
- **Decoder**: Generates the output sequence, using both self-attention and encoder-decoder attention mechanisms
- **Multi-Head Attention**: Allows the model to focus on different parts of the input sequence simultaneously
- **Positional Encoding**: Adds information about the position of tokens in the sequence
- **Feed-Forward Networks**: Process the attention output through fully connected layers
- **Residual Connections & Layer Normalization**: Help with gradient flow and training stability

This architecture has revolutionized sequence processing by eliminating recurrence and enabling highly parallelized training.

```mermaid
flowchart LR
    Input("Input Embeddings") --> AddPos("+ Positional Encoding")
    AddPos --> EncoderStack("Encoder Stack")
    Output("Output Embeddings") --> AddPosOut("+ Positional Encoding")
    AddPosOut --> DecoderStack("Decoder Stack")
    EncoderStack --> DecoderStack
    DecoderStack --> Linear("Linear Layer")
    Linear --> Softmax("Softmax")
    Softmax --> FinalOutput("Output Probabilities")
    
    subgraph "Encoder Block × N"
        EncIn("Input") --> MultiHead1("Multi-Head Self-Attention")
        MultiHead1 --> AddNorm1("Add & Norm")
        EncIn -.-> AddNorm1
        AddNorm1 --> FFN1("Feed Forward")
        FFN1 --> AddNorm2("Add & Norm")
        AddNorm1 -.-> AddNorm2
    end
    
    subgraph "Decoder Block × N"
        DecIn("Input") --> MaskedMultiHead("Masked Multi-Head Self-Attention")
        MaskedMultiHead --> AddNorm3("Add & Norm")
        DecIn -.-> AddNorm3
        AddNorm3 --> MultiHead2("Multi-Head Cross-Attention")
        MultiHead2 --> AddNorm4("Add & Norm")
        AddNorm3 -.-> AddNorm4
        AddNorm4 --> FFN2("Feed Forward")
        FFN2 --> AddNorm5("Add & Norm")
        AddNorm4 -.-> AddNorm5
    end
```

## Full Math

Notation
- Source tokens: $x_{1:n}$, target tokens: $y_{1:m}$ (with BOS at $y_1$ during training)
- Embedding matrix: $E\in\mathbb{R}^{|\mathcal{V}|\times d_{model}}$, positional encodings $P\in\mathbb{R}^{L_{max}\times d_{model}}$
- Heads: $H$ heads with $d_k=d_v=d_{model}/H$
- LayerNorm is applied as in the original paper: $\mathrm{LayerNorm}(x + \mathrm{Sublayer}(x))$ (post-norm)

### Encoder: from input tokens to contextual states
1) Input embeddings (optionally scaled as in the paper):
$$Z^{(0)}_i = \sqrt{d_{model}}\,E[x_i] + P[i], \quad i=1,\dots,n,$$
collect all rows into $Z^{(0)}\in\mathbb{R}^{n\times d_{model}}$.

2) Multi-head self-attention at layer $\ell\in\{1,\dots,N\}$ (for each head $h=1,\dots,H$):
$$Q_h = Z^{(\ell-1)} W^{Q}_{h},\quad K_h = Z^{(\ell-1)} W^{K}_{h},\quad V_h = Z^{(\ell-1)} W^{V}_{h},$$
with $W^{Q}_{h},W^{K}_{h}\in\mathbb{R}^{d_{model}\times d_k}$ and $W^{V}_{h}\in\mathbb{R}^{d_{model}\times d_v}$.
$$A_h = \mathrm{softmax}\!\left( \frac{Q_h K_h^\top}{\sqrt{d_k}} \right) V_h \;\in\mathbb{R}^{n\times d_v}.$$
Concatenate heads and project:
$$\mathrm{MHA}\big(Z^{(\ell-1)}\big) = \big[ A_1\,\Vert\,\cdots\,\Vert\,A_H \big] W^{O},\quad W^{O}\in\mathbb{R}^{(H d_v)\times d_{model}}.$$
Residual + LayerNorm:
$$\tilde{Z}^{(\ell)} = \mathrm{LayerNorm}\!\big( Z^{(\ell-1)} + \mathrm{MHA}(Z^{(\ell-1)}) \big).$$

3) Position-wise feed-forward (same MLP at each position):
$$\mathrm{FFN}(x) = \max(0,\, x W_1 + b_1) W_2 + b_2,$$
with $W_1\in\mathbb{R}^{d_{model}\times d_{ff}}$, $W_2\in\mathbb{R}^{d_{ff}\times d_{model}}$.
Residual + LayerNorm:
$$Z^{(\ell)} = \mathrm{LayerNorm}\!\big( \tilde{Z}^{(\ell)} + \mathrm{FFN}(\tilde{Z}^{(\ell)}) \big).$$

4) Encoder output:
$$H = Z^{(N)}\in\mathbb{R}^{n\times d_{model}}.$$

### Decoder: from target prefix and encoder states to next-token distribution
1) Target embeddings:
$$Y^{(0)}_t = \sqrt{d_{model}}\,E[y_t] + P[t], \quad t=1,\dots,m,$$
collect $Y^{(0)}\in\mathbb{R}^{m\times d_{model}}$.

2) Masked self-attention at layer $\ell$ (causal mask $M$ with $M_{ij}=-\infty$ if $i<j$, else $0$):
For each head $h$:
$$Q_h = Y^{(\ell-1)} W^{Q}_{h},\quad K_h = Y^{(\ell-1)} W^{K}_{h},\quad V_h = Y^{(\ell-1)} W^{V}_{h},$$
$$A^{\text{self}}_h = \mathrm{softmax}\!\left( \frac{Q_h K_h^\top + M}{\sqrt{d_k}} \right) V_h.$$
Combine heads and project as before to obtain $\mathrm{MHA}^{\text{mask}}(Y^{(\ell-1)})$.
Residual + LayerNorm:
$$\tilde{Y}^{(\ell)} = \mathrm{LayerNorm}\!\big( Y^{(\ell-1)} + \mathrm{MHA}^{\text{mask}}(Y^{(\ell-1)}) \big).$$

3) Encoder–decoder (cross) attention at layer $\ell$ (queries from decoder, keys/values from encoder $H$):
For each head $h$:
$$Q_h = \tilde{Y}^{(\ell)} W^{Q,c}_{h},\quad K_h = H W^{K,c}_{h},\quad V_h = H W^{V,c}_{h},$$
$$A^{\text{cross}}_h = \mathrm{softmax}\!\left( \frac{Q_h K_h^\top}{\sqrt{d_k}} \right) V_h.$$
Project concatenated heads:
$$\mathrm{MHA}^{\text{cross}}(\tilde{Y}^{(\ell)}, H) = \big[ A^{\text{cross}}_1\,\Vert\,\cdots\,\Vert\,A^{\text{cross}}_H \big] W^{O,c}.$$
Residual + LayerNorm:
$$\hat{Y}^{(\ell)} = \mathrm{LayerNorm}\!\big( \tilde{Y}^{(\ell)} + \mathrm{MHA}^{\text{cross}}(\tilde{Y}^{(\ell)}, H) \big).$$

4) Position-wise FFN at layer $\ell$:
$$Y^{(\ell)} = \mathrm{LayerNorm}\!\big( \hat{Y}^{(\ell)} + \mathrm{FFN}(\hat{Y}^{(\ell)}) \big).$$

5) Output logits and distribution (per position $t$):
$$o_t = Y^{(N)}_t W^{\text{out}} + b,\qquad p(y_t\mid y_{<t}, x_{1:n}) = \mathrm{softmax}(o_t).$$
Often $W^{\text{out}}$ is tied with $E^\top$.

Training uses teacher forcing with the same causal mask; inference generates tokens autoregressively using the decoder state and cross-attention to $H$. Mechanism

The core innovation of the Transformer is its attention mechanism, specifically the "Scaled Dot-Product Attention".

### Scaled Dot-Product Attention


The attention function maps a query and a set of key-value pairs to an output. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed using a compatibility function of the query with the corresponding key.

Mathematically, this is expressed as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ is the matrix of queries
- $K$ is the matrix of keys
- $V$ is the matrix of values
- $d_k$ is the dimension of the keys

The scaling factor $\sqrt{d_k}$ prevents the dot products from growing too large in magnitude, which would push the softmax function into regions with extremely small gradients.

```mermaid
graph TB
    Q["Q: Queries"] --> MatMul1["MatMul"]
    K["K: Keys"] --> Transpose["Transpose"]
    Transpose --> MatMul1
    MatMul1 --> Scale["Scale by 1/√dk"]
    Scale --> Mask["Optional Mask<br/>(decoder only)"]
    Mask --> Softmax["Softmax"]
    Softmax --> MatMul2["MatMul"]
    V["V: Values"] --> MatMul2
    MatMul2 --> Output["Attention Output"]
```

### Multi-Head Attention

Rather than performing a single attention function, the Transformer uses multi-head attention:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \text{head}_2, ..., \text{head}_h)W^O$$

Where each head is calculated as:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

The projections are parameter matrices $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$ and $W^O \in \mathbb{R}^{hd_v \times d_{model}}$.

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions, providing more diverse features for subsequent layers.

```mermaid
graph TD
    Input["Input"] --> SplitHeads["Linear Projections"]
    SplitHeads -->|"WQ1"| Q1["Q1"]
    SplitHeads -->|"WK1"| K1["K1"]
    SplitHeads -->|"WV1"| V1["V1"]
    Q1 --> Attn1["Attention Head 1"]
    K1 --> Attn1
    V1 --> Attn1
    
    SplitHeads -->|"WQ2"| Q2["Q2"]
    SplitHeads -->|"WK2"| K2["K2"]
    SplitHeads -->|"WV2"| V2["V2"]
    Q2 --> Attn2["Attention Head 2"]
    K2 --> Attn2
    V2 --> Attn2
    
    SplitHeads -->|"..."| Qn["..."]
    SplitHeads -->|"..."| Kn["..."]
    SplitHeads -->|"..."| Vn["..."]
    
    SplitHeads -->|"WQh"| Qh["Qh"]
    SplitHeads -->|"WKh"| Kh["Kh"]
    SplitHeads -->|"WVh"| Vh["Vh"]
    Qh --> Attnh["Attention Head h"]
    Kh --> Attnh
    Vh --> Attnh
    
    Attn1 --> Concat["Concatenate"]
    Attn2 --> Concat
    Attnh --> Concat
    Concat --> Linear["Linear Projection WO"]
    Linear --> Output["Multi-Head Output"]
```

## Positional Encoding

Since the Transformer contains no recurrence or convolution, it needs some way to incorporate the order of the sequence. This is achieved through positional encodings which are added to the input embeddings.

The positional encodings have the same dimension as the embeddings, allowing them to be summed. The formula used is:

$$PE_{(pos,2i)} = \sin(pos/10000^{2i/d_{model}})$$
$$PE_{(pos,2i+1)} = \cos(pos/10000^{2i/d_{model}})$$

Where $pos$ is the position and $i$ is the dimension. Each dimension of the positional encoding corresponds to a sinusoid with different frequencies.

The wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$. This allows the model to easily learn to attend by relative positions, since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

```mermaid
graph LR
    Words["Word Embeddings"] --> Add["+ Addition"]
    PositionalEnc["Positional Encodings"] --> Add
    Add --> Output["Input to Transformer"]
    
    subgraph "Positional Encoding Generation"
        Pos["Position index p"] --> SinCalc["sin(p/10000^(2i/d))"]
        Pos --> CosCalc["cos(p/10000^(2i/d))"]
        SinCalc --> EvenDim["Even dimensions"]
        CosCalc --> OddDim["Odd dimensions"]
        EvenDim --> PosEncVec["Position Encoding Vector"]
        OddDim --> PosEncVec
    end
```

The plot above shows how positional encodings vary with position (x-axis) and dimension (y-axis). The pattern enables the model to determine the relative position of words in a sequence.

![positional encoding](https://d33wubrfki0l68.cloudfront.net/ef81ee3018af6ab6f23769031f8961afcdd67c68/3358f/img/transformer_architecture_positional_encoding/positional_encoding.png)

## Encoder Structure

The encoder consists of a stack of $N$ identical layers (typically 6 in the original paper). Each layer has two sub-layers:

1. **Multi-Head Self-Attention mechanism**
2. **Position-wise Fully Connected Feed-Forward Network**

Around each sub-layer is a residual connection, followed by layer normalization. Mathematically:

$$\text{LayerNorm}(x + \text{Sublayer}(x))$$

Where $\text{Sublayer}(x)$ is the function implemented by the sub-layer itself.

```mermaid
graph TD
    InputEmb["Input"] --> SelfAttn["Multi-Head<br/>Self-Attention"]
    InputEmb -->|"Residual Connection"| Add1["Add"]
    SelfAttn --> Add1
    Add1 --> Norm1["Layer Norm"]
    
    Norm1 --> FFN["Position-wise<br/>Feed-Forward Network"]
    Norm1 -->|"Residual Connection"| Add2["Add"]
    FFN --> Add2
    Add2 --> Norm2["Layer Norm"]
    Norm2 --> Output["Output"]
    
    subgraph "Encoder Repeated N Times"
        Enc1["Encoder Layer 1"] --> Enc2["Encoder Layer 2"]
        Enc2 --> EllipseEnc["..."]
        EllipseEnc --> EncN["Encoder Layer N"]
    end
```

### Feed-Forward Network

The Feed-Forward Network (FFN) consists of two linear transformations with a ReLU activation in between:

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

Each position is processed identically but with different parameters, making it effectively a position-wise feed-forward network. The dimensionality typically follows the pattern:
- Input dimension: $d_{model}$ (e.g., 512)
- Inner-layer dimension: $d_{ff}$ (e.g., 2048)
- Output dimension: $d_{model}$ (e.g., 512)

## Decoder Structure

The decoder also consists of a stack of $N$ identical layers, but each has three sub-layers:

1. **Masked Multi-Head Self-Attention**
2. **Multi-Head Encoder-Decoder Attention**
3. **Position-wise Feed-Forward Network**

The masked attention in the first sub-layer ensures that predictions for position $i$ can depend only on the known outputs at positions less than $i$. This masking is achieved by:

$$\text{Mask}(Q, K, V) = \text{softmax}\left(\frac{QK^T + M}{\sqrt{d_k}}\right)V$$

Where $M$ is a matrix with:
$$M_{ij} = \begin{cases} 
0 & \text{if } i \geq j \\ 
-\infty & \text{if } i < j
\end{cases}$$

```mermaid
graph TD
    InputEmb["Input"] --> MaskedAttn["Masked Multi-Head<br/>Self-Attention"]
    InputEmb -->|"Residual Connection"| Add1["Add"]
    MaskedAttn --> Add1
    Add1 --> Norm1["Layer Norm"]
    
    Norm1 --> CrossAttn["Multi-Head<br/>Encoder-Decoder Attention"]
    EncoderOut["Encoder Output"] --> CrossAttn
    Norm1 -->|"Residual Connection"| Add2["Add"]
    CrossAttn --> Add2
    Add2 --> Norm2["Layer Norm"]
    
    Norm2 --> FFN["Position-wise<br/>Feed-Forward Network"]
    Norm2 -->|"Residual Connection"| Add3["Add"]
    FFN --> Add3
    Add3 --> Norm3["Layer Norm"]
    Norm3 --> Output["Output"]
    
    subgraph "Masked Self-Attention"
        Tokens["Output Tokens<br/>(so far)"] --> Mask["Apply Future Mask"]
        Mask --> SelfAttention["Self-Attention<br/>Mechanism"]
    end
```

The second attention layer performs multi-head attention where:
- Queries come from the previous decoder layer
- Keys and values come from the encoder output

This allows every position in the decoder to attend to all positions in the input sequence, implementing the encoder-decoder attention mechanism.

## Limitations and Variants

Despite its strengths, the Transformer has some limitations:

1. **Quadratic Complexity**: The self-attention mechanism has O(n²) complexity with respect to sequence length, limiting its application to very long sequences

2. **Fixed Context Window**: Most implementations have a maximum sequence length, beyond which they cannot process

3. **Lack of Built-in Inductive Bias**: Unlike CNNs (locality) and RNNs (sequentiality), Transformers have minimal inductive bias about the structure of language or sequences

### Notable Variants

```mermaid
graph TD
    Original["Original Transformer<br/>O(n²) complexity"] --> TXL["Transformer-XL<br/>Segment recurrence"]
    Original --> Reformer["Reformer<br/>LSH attention<br/>O(n log n)"]
    Original --> Longformer["Longformer<br/>Sparse attention<br/>O(n)"]
    Original --> Linformer["Linformer<br/>Projected attention<br/>O(n)"]
    Original --> Performer["Performer<br/>FAVOR+ kernel<br/>O(n)"]
    Reformer --> Routing["Routing Transformer<br/>Clustered attention"]
    Longformer --> BigBird["BigBird<br/>Global + local + random"]
    TXL --> Compressive["Compressive Transformer<br/>Memory compression"]
    Original --> Sparse["Sparse Transformer<br/>Sparse factorizations"]
```

Several variants have been proposed to address these limitations:

| Variant | Key Innovation | Complexity |
|---------|----------------|------------|
| [Transformer-XL](https://arxiv.org/abs/1901.02860) | Segment-level recurrence for longer contexts | O(n²) with cached states |
| [Reformer](https://arxiv.org/abs/2001.04451) | Locality-sensitive hashing for efficient attention | O(n log n) |
| [Longformer](https://arxiv.org/abs/2004.05150) | Sliding window attention with global tokens | O(n) |
| [Linformer](https://arxiv.org/abs/2006.04768) | Projected attention for linear complexity | O(n) |
| [Performer](https://arxiv.org/abs/2009.14794) | FAVOR+ approximation for efficient attention | O(n) |

These variants maintain the core principles of the Transformer while addressing specific limitations, further expanding the applicability of attention-based architectures.

## Conclusion

The Transformer architecture has fundamentally changed the landscape of sequence modeling and natural language processing. Its key innovations include:

- **Parallelization**: Enabling efficient training on massive datasets
- **Attention Mechanism**: Providing direct modeling of relationships between all elements in a sequence
- **Scalable Architecture**: Supporting models from millions to billions of parameters

These characteristics have made Transformers the foundation for most state-of-the-art NLP models since 2018, including BERT, GPT, T5, and others. The architecture continues to evolve, with ongoing research addressing its limitations and extending its capabilities to new domains beyond natural language processing, such as computer vision, speech recognition, and reinforcement learning.

The Transformer represents one of the most significant architectural innovations in deep learning, demonstrating that attention mechanisms alone can provide powerful sequence modeling capabilities without the need for recurrence or convolution.

## References

1. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762). *Neural Information Processing Systems (NeurIPS)*.

2. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2018). [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805). *arXiv preprint*.

3. Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan, J., Dhariwal, P., ... & Amodei, D. (2020). [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165). *Neural Information Processing Systems (NeurIPS)*.

4. Alammar, J. (2018). [The Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/). *Blog Post*.

5. Tay, Y., Dehghani, M., Bahri, D., & Metzler, D. (2020). [Efficient Transformers: A Survey](https://arxiv.org/abs/2009.06732). *arXiv preprint*.

6. Lin, Z., Feng, M., Santos, C. N. D., Yu, M., Xiang, B., Zhou, B., & Bengio, Y. (2017). [A Structured Self-attentive Sentence Embedding](https://arxiv.org/abs/1703.03130). *International Conference on Learning Representations (ICLR)*.